## testing PCR images with fixed setters

In [1]:
# from grpc4bmi.bmi_client_singularity import BmiClientSingularity

# model = BmiClientSingularity('/home/avandervee3/pcr_setters_fixed.sif', work_dir='/tmp')
# print(model.get_component_name())
# del model

In [2]:
import matplotlib.pyplot as plt
from cartopy import crs
from cartopy import feature as cfeature
from rich import print
import pandas as pd
import xarray as xr
from pathlib import Path
from datetime import datetime
from ipywidgets import IntProgress
from IPython.display import display
import fiona
import shapely.geometry
from pyproj import Geod

import ewatercycle.forcing
import ewatercycle.models
import ewatercycle.parameter_sets
from ewatercycle.container import ContainerImage
from tqdm import tqdm

/opt/conda/envs/ewatercycle2/lib/python3.12/site-packages/esmvalcore/experimental/_warnings.py:13: UserWarning: 
  Thank you for trying out the new ESMValCore API.
  Note that this API is experimental and may be subject to change.
  More info: https://github.com/ESMValGroup/ESMValCore/issues/498


In [3]:
# Chatly_station_latitude = 42.34332908492399  # Amu Darya near Chatly
# Chatly_station_longitude = 59.627516175820965 

Chatly_station_latitude = 42.34332908492399-0.5 # Amu Darya near Chatly
Chatly_station_longitude = 59.627516175820965+0.9    # Through trial and error, to get modelled discharge closer to observed

Kerki_station_latitude = 37.8396310038444 #Amu Darya near Kerki
Kerki_station_longitude = 65.23703868931334

#Tyumen_station_latitude = 44.01445789449254 # Syr Darya near Tyumen, google maps
#Tyumen_station_longitude = 67.02866313732494

#Tyumen_station_latitude = 44.05 # Syr Darya near Tyumen, GRDC coords
#Tyumen_station_longitude = 67.05

Tyumen_station_latitude = 44.05-0.1 # Syr Darya near Tyumen, through trial and error
Tyumen_station_longitude = 67.05

Kazalinsk_station_latitude = 45.739988222442456, #Syr Darya near Kazalinsk
Kazalinsk_station_longitude = 62.115993992559744

Dushanbe_station_latitude = 38.76042
Dushanbe_staion_longitude = 68.81458



In [4]:
# set start and end date of the experiment, overwrites .ini settings
experiment_start_date = "2010-01-01T00:00:00Z"
#experiment_end_date = "1992-12-31T00:00:00Z" #was 1995-12-31T00:00:00Z, shorter for testing
#experiment_end_date = "1990-02-28T00:00:00Z" #2 months, for testing
experiment_end_date = "2010-03-31T00:00:00Z" #2 months, for testing

In [5]:


pcr_glob_directory = Path("/data/shared/parameter-sets/pcrglobwb_global")  #GlobalOption uit .ini

prepared_PCRGlob_forcing = (
    Path("/home/avandervee3/MSc_AralSea/book/thesis_projects/MSc/2025_Q1_AndreVanDerVeen_CEG/work_in_progress/Test_Aral/forcing_0419")
    / "AralSeaBasin"
    / "pcrglobwb"
    / "work/diagnostic/script"
)##MeteoOptions uit .ini

prepared_PCRGlob_forcing_CMIP = (
    Path("/home/avandervee3/MSc_AralSea/book/thesis_projects/MSc/2025_Q1_AndreVanDerVeen_CEG/work_in_progress/Comparison/forcing_CMIP_5570")
    / "AralSeaBasin"
    / "CMIP6" / "historic"/"PCRGlobWB"
    / "work/diagnostic/script"
)##MeteoOptions uit .ini



In [6]:


parameter_set_cmip = ewatercycle.parameter_sets.ParameterSet(
    name="custom_parameter_set",
    directory=pcr_glob_directory,
    config= "/home/avandervee3/MSc_AralSea/book/thesis_projects/MSc/2025_Q1_AndreVanDerVeen_CEG/work_in_progress/Comparison/Reference_05min_agri_copy.ini",
    target_model="pcrglobwb",
    supported_model_versions={"globwb"},
)


# parameter_set_cmip_no_agri = ewatercycle.parameter_sets.ParameterSet(
#     name="custom_parameter_set",
#     directory=pcr_glob_directory,
#     config= Path.cwd() / "Reference_05min_no_agri.ini",
#     target_model="pcrglobwb",
#     supported_model_versions={"fixed"},
# )



In [7]:
forcing = ewatercycle.forcing.sources["PCRGlobWBForcing"].load(
    directory=prepared_PCRGlob_forcing,
)




print(forcing)

PCRGlobWBForcing(
    start_time='2004-01-01T00:00:00Z',
    end_time='2019-12-31T00:00:00Z',
    directory=PosixPath('/home/avandervee3/MSc_AralSea/book/thesis_projects/MSc/2025_Q1_AndreVanDerVeen_CEG/work_in
_progress/Test_Aral/forcing_0419/AralSeaBasin/pcrglobwb/work/diagnostic/script'),
    shape=PosixPath('/home/avandervee3/MSc_AralSea/book/thesis_projects/MSc/2025_Q1_AndreVanDerVeen_CEG/work_in_pro
gress/Test_Aral/forcing_0419/AralSeaBasin/pcrglobwb/work/diagnostic/script/AralSeaBasin.shp'),
    filenames={},
    precipitationNC='pcrglobwb_OBS6_ERA5_reanaly_1_day_pr_2004-2019_AralSeaBasin.nc',
    temperatureNC='pcrglobwb_OBS6_ERA5_reanaly_1_day_tas_2004-2019_AralSeaBasin.nc'
)

In [8]:
my_image = ContainerImage('/home/avandervee3/ewatercycle_pcr_globwb.sif')
my_image.version

'globwb'

In [9]:
reference = ewatercycle.models.PCRGlobWB(
    parameter_set=parameter_set_cmip,
    forcing=forcing,
    bmi_image=my_image
)

print(reference)

PCRGlobWB(
    parameter_set=ParameterSet(
        name='custom_parameter_set',
        directory=PosixPath('/data/shared/parameter-sets/pcrglobwb_global'),
        config=PosixPath('/home/avandervee3/MSc_AralSea/book/thesis_projects/MSc/2025_Q1_AndreVanDerVeen_CEG/work_i
n_progress/Comparison/Reference_05min_agri_copy.ini'),
        doi='N/A',
        target_model='pcrglobwb',
        supported_model_versions={'globwb'},
        downloader=None
    ),
    forcing=PCRGlobWBForcing(
        start_time='2004-01-01T00:00:00Z',
        end_time='2019-12-31T00:00:00Z',
        directory=PosixPath('/home/avandervee3/MSc_AralSea/book/thesis_projects/MSc/2025_Q1_AndreVanDerVeen_CEG/wor
k_in_progress/Test_Aral/forcing_0419/AralSeaBasin/pcrglobwb/work/diagnostic/script'),
        shape=PosixPath('/home/avandervee3/MSc_AralSea/book/thesis_projects/MSc/2025_Q1_AndreVanDerVeen_CEG/work_in
_progress/Test_Aral/forcing_0419/AralSeaBasin/pcrglobwb/work/diagnostic/script/AralSeaBasin.shp'),
        filenames={},
        precipitationNC='pcrglobwb_OBS6_ERA5_reanaly_1_day_pr_2004-2019_AralSeaBasin.nc',
        temperatureNC='pcrglobwb_OBS6_ERA5_reanaly_1_day_tas_2004-2019_AralSeaBasin.nc'
    )
)

In [10]:
reference_config, reference_dir = reference.setup(
    start_time = experiment_start_date,
    end_time = experiment_end_date,
    max_spinups_in_years=0
)
reference_config, reference_dir

TimeoutError: Couldn't spawn container within allocated time limit (300 seconds). You may try pulling the docker image with `apptainer build /home/avandervee3/ewatercycle_pcr_globwb.sif docker:///home/avandervee3/ewatercycle_pcr:globwb` and then try again.

In [11]:
from grpc4bmi.bmi_client_singularity import BmiClientSingularity

model = BmiClientSingularity(
    image="/home/avandervee3/ewatercycle_pcr_globwb.sif",
    work_dir="/tmp",
)

print(model.get_component_name())
del model   # shut down container + server

KeyboardInterrupt: 

In [ ]:
print(reference.parameters)

refence_para = reference.parameters

# Convert ISO 8601 strings to datetime objects
start_time = datetime.strptime(experiment_start_date, '%Y-%m-%dT%H:%M:%SZ')
end_time = datetime.strptime(experiment_end_date, '%Y-%m-%dT%H:%M:%SZ')

# Calculate the number of days for the progression bar
delta = end_time - start_time
number_of_days = delta.days
print(f"Number of days to model: {number_of_days}")

In [ ]:
print("Using config:", reference_config)

In [ ]:
reference.initialize(reference_config)

In [ ]:
time = pd.date_range(reference.start_time_as_isostr, reference.end_time_as_isostr)
# timeseries = pd.DataFrame(
#     index=pd.Index(time, name="time"), columns=["reference", "experiment1", "experiment2"]
# )
# timeseries.head()

In [ ]:
Stations_timeseries = pd.DataFrame(
    index=pd.Index(time, name="time"), columns=["Chatly", "Kerki", "Tyumen", "Kazalinsk", "Dushanbe"]
)
Stations_timeseries.head()

Stations_timeseries_experiment_no_agri = Stations_timeseries.copy();
Stations_timeseries_experiment_RoutingWave = Stations_timeseries.copy();

In [ ]:
# Progress bar, since this can take a while
f = IntProgress(min=0, max=number_of_days) # instantiate the bar
display(f) # display the bar

for _ in tqdm(range(number_of_days), desc="Running model"):
    
    reference.update()

    # # Track discharge at station location
    # discharge_at_Chatly = reference.get_value_at_coords(
    #     "discharge", lat=[Chatly_station_latitude], lon=[Chatly_station_longitude]
    # )
    # time = reference.time_as_isostr
    # Stations_timeseries.loc[time, "Chatly"] = discharge_at_Chatly[0]

    #  # Track discharge at station location Kerki
    # discharge_at_Kerki = reference.get_value_at_coords(
    #     "discharge", lat=[Kerki_station_latitude], lon=[Kerki_station_longitude]
    # )
    # time = reference.time_as_isostr
    # Stations_timeseries.loc[time, "Kerki"] = discharge_at_Kerki[0]

    # # Track discharge at station Tyumen
    # discharge_at_Tyumen = reference.get_value_at_coords(
    #     "discharge", lat=[Tyumen_station_latitude], lon=[Tyumen_station_longitude]
    # )
    # time = reference.time_as_isostr
    # Stations_timeseries.loc[time, "Tyumen"] = discharge_at_Tyumen[0]

    # # Track discharge at station location Karalinsk
    # discharge_at_Kazalinsk = reference.get_value_at_coords(
    #     "discharge", lat=[Kazalinsk_station_latitude], lon=[Kazalinsk_station_longitude]
    # )
    # time = reference.time_as_isostr
    # Stations_timeseries.loc[time, "Kazalinsk"] = discharge_at_Kazalinsk[0]

    # # Track discharge at station location Dushanbe
    # discharge_at_Dushanbe = reference.get_value_at_coords(
    #     "discharge", lat=[Dushanbe_station_latitude], lon=[Dushanbe_staion_longitude]
    # )
    # time = reference.time_as_isostr
    # Stations_timeseries.loc[time, "Dushanbe"] = discharge_at_Dushanbe[0]    




print("Model run finished!")

In [ ]:
reference.get_value_as_xarray("discharge")